# **Durability — RUL DATASET GENERATION**

## Prerequisite

Run [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) $\to$
[`02_train_pce.ipynb`](02_train_pce.ipynb) $\to$
[`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb) $\to$
[`04_train_nn.ipynb`](04_train_nn.ipynb) first — this notebook loads the global NN
(`lambda 1`/`lambda 2` vs. `(fck, rh, cov, t)`) that stage 4 writes.

## What this notebook does

Fixes a single design point `(fck, rh, cov)` and sweeps a list of time steps through the trained
NN, giving $\lambda_1(t)$ and $\lambda_2(t)$ at each one **directly** — no need to pick a
per-time-step PCE first. $\lambda_3$ and $\lambda_4$ barely move across the design space, which is
why they were never modelled by the NN — they're fixed here instead, to a value you set or, by
default, the mean over the NN's own training dataset.

At each time step, `generate_rul_dataset_durability` builds a `GlamFKML(lam1, lam2, lam3, lam4)` and
draws `n_glam_samples` Monte Carlo realisations of the state limit function $g$ = cover −
carbonation depth — the raw material for the spaghetti plot and the RUL analysis in
[`05_plot_rul_analysis.ipynb`](05_plot_rul_analysis.ipynb).

## 1. Libraries

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

import importlib
import functions

# Recarrega functions.py ao executar esta celula novamente.
importlib.reload(functions)
from functions import *

e:\github\2024-1_victor_hugo_renata_maria\.venv\Lib\site-packages\UQpy\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Config

`n_latent_samples`, `installation_year`, `co2_scenario`, `cement_type` and `exposure_conditions`
must match [`04_train_nn.ipynb`](04_train_nn.ipynb) — it names the NN model files being loaded.

`lambda3_fixed`/`lambda4_fixed` left as `None` fall back to the mean of `lambda 3`/`lambda 4` over
the NN training dataset; set either explicitly to override.

In [2]:
n_latent_samples    = 2500      # must match stage 4 — it names the NN model files being loaded
installation_year   = 1980
co2_scenario        = "SSP2-4.5"
cement_type         = 3
exposure_conditions = 2

fck_fixed = 25.0   # fixed compressive strength for the whole sweep (MPa)
rh_fixed  = 55.0   # fixed relative humidity for the whole sweep (%)
cov_fixed = 25.0   # fixed cover for the whole sweep (mm)
times     = np.linspace(0, 100, 20, endpoint=True)  # time steps to sweep through the NN — any grid, not just the training one

lambda3_fixed = 0.201702   # None = mean of lambda 3 over the NN training dataset
lambda4_fixed = 0.136161   # None = mean of lambda 4 over the NN training dataset

n_glam_samples = 50000   # Monte Carlo samples drawn from the GLD at each time step

print(f"Sweeping t = {times.min()}..{times.max()} years at fck={fck_fixed}, rh={rh_fixed}, cov={cov_fixed}")

# Arquivos de entrada usados para gerar o dataset de RUL.
input_dir = Path('.').resolve()
input_tag = f'install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'
rul_input_files = {
    'Modelo NN - lambda 1': input_dir / f'{n_latent_samples}_nn_lambda_1_model_durability_{input_tag}.pkl',
    'Modelo NN - lambda 2': input_dir / f'{n_latent_samples}_nn_lambda_2_model_durability_{input_tag}.pkl',
    'Scaler NN': input_dir / f'{n_latent_samples}_nn_scaler_durability_{input_tag}.pkl',
}
if lambda3_fixed is None or lambda4_fixed is None:
    rul_input_files['Dataset NN - medias de lambda 3/4'] = input_dir / f'{n_latent_samples}_dataset_nn_durability_{input_tag}.pkl'

print('\n[RUL] Arquivos que serao carregados:')
for label, path in rul_input_files.items():
    status = 'ENCONTRADO' if path.is_file() else 'AUSENTE'
    print(f'  [{status}] {label}: {path}')
if lambda3_fixed is not None and lambda4_fixed is not None:
    print('[RUL] Lambda 3/4 definidos acima; o dataset NN nao sera carregado.')


Sweeping t = 0.0..100.0 years at fck=25.0, rh=55.0, cov=25.0

[RUL] Arquivos que serao carregados:
  [ENCONTRADO] Modelo NN - lambda 1: E:\github\2024-1_victor_hugo_renata_maria\durability\2500_nn_lambda_1_model_durability_install_1980_cement_3_exposure_2_co2_SSP2-4.5.pkl
  [ENCONTRADO] Modelo NN - lambda 2: E:\github\2024-1_victor_hugo_renata_maria\durability\2500_nn_lambda_2_model_durability_install_1980_cement_3_exposure_2_co2_SSP2-4.5.pkl
  [ENCONTRADO] Scaler NN: E:\github\2024-1_victor_hugo_renata_maria\durability\2500_nn_scaler_durability_install_1980_cement_3_exposure_2_co2_SSP2-4.5.pkl
[RUL] Lambda 3/4 definidos acima; o dataset NN nao sera carregado.


## 3. Predict lambda 1/2, fix lambda 3/4, and draw the GLD samples

In [3]:
print("="*60)
print("GENERATING THE RUL DATASET")
print("="*60)

result = generate_rul_dataset_durability(
                                            fck=fck_fixed,
                                            rh=rh_fixed,
                                            cov=cov_fixed,
                                            times=times,
                                            n_latent_samples=n_latent_samples,
                                            installation_year=installation_year,
                                            cement_type=cement_type,
                                            exposure_conditions=exposure_conditions,
                                            co2_scenario=co2_scenario,
                                            lambda3_fixed=lambda3_fixed,
                                            lambda4_fixed=lambda4_fixed,
                                            n_glam_samples=n_glam_samples,
                                            input_dir=input_dir,
                                            output_dir='.',
                                         )

lambda_df = result['lambda_df']
samples   = result['samples']
print(f"\nSamples shape: {samples.shape}  (n_glam_samples x len(times))")
lambda_df

GENERATING THE RUL DATASET

----------------------------------------
GENERATING RUL DATASET AT fck=25.0, rh=55.0, cov=25.0
----------------------------------------
[PKL] Carregando: E:\github\2024-1_victor_hugo_renata_maria\durability\2500_nn_lambda_1_model_durability_install_1980_cement_3_exposure_2_co2_SSP2-4.5.pkl
[PKL] Carregado: E:\github\2024-1_victor_hugo_renata_maria\durability\2500_nn_lambda_1_model_durability_install_1980_cement_3_exposure_2_co2_SSP2-4.5.pkl
[PKL] Carregando: E:\github\2024-1_victor_hugo_renata_maria\durability\2500_nn_lambda_2_model_durability_install_1980_cement_3_exposure_2_co2_SSP2-4.5.pkl
[PKL] Carregado: E:\github\2024-1_victor_hugo_renata_maria\durability\2500_nn_lambda_2_model_durability_install_1980_cement_3_exposure_2_co2_SSP2-4.5.pkl
[PKL] Carregando: E:\github\2024-1_victor_hugo_renata_maria\durability\2500_nn_scaler_durability_install_1980_cement_3_exposure_2_co2_SSP2-4.5.pkl
[PKL] Carregado: E:\github\2024-1_victor_hugo_renata_maria\durability\2

,fck,rh,cov,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,25.0,55.0,25.0,0.000000,24.948891,0.392639,0.201702,0.136161
1,25.0,55.0,25.0,5.263158,21.710262,0.354084,0.201702,0.136161
2,25.0,55.0,25.0,10.526316,18.244565,0.318874,0.201702,0.136161
3,25.0,55.0,25.0,15.789474,14.342489,0.297105,0.201702,0.136161
4,25.0,55.0,25.0,21.052632,10.322995,0.280465,0.201702,0.136161
5,25.0,55.0,25.0,26.315789,7.177633,0.263090,0.201702,0.136161
6,25.0,55.0,25.0,31.578947,5.279880,0.244220,0.201702,0.136161
7,25.0,55.0,25.0,36.842105,3.695017,0.230375,0.201702,0.136161
8,25.0,55.0,25.0,42.105263,2.375243,0.222591,0.201702,0.136161
9,25.0,55.0,25.0,47.368421,1.154969,0.213192,0.201702,0.136161


## 4. Sanity check

In [4]:
summary = pd.DataFrame({
                           'Time (years)': result['times'],
                           'Sample mean':  samples.mean(axis=0),
                           'Sample std':   samples.std(axis=0),
                           'P(g <= 0)':    (samples <= 0).mean(axis=0),
                        })
summary

,Time (years),Sample mean,Sample std,P(g <= 0)
0,0.000000,25.078865,3.559639,0.00000
1,5.263158,21.854388,3.947230,0.00000
2,10.526316,18.404606,4.383092,0.00000
3,15.789474,14.514257,4.704241,0.00010
4,21.052632,10.504953,4.983329,0.01288
5,26.315789,7.371608,5.312456,0.08048
6,31.578947,5.488843,5.722930,0.17356
7,36.842105,3.916538,6.066858,0.26868
8,42.105263,2.604511,6.279020,0.34888
9,47.368421,1.394345,6.555844,0.42516


In [5]:
summary_5mm = pd.DataFrame({
                              'Time (years)': result['times'],
                              'Sample mean':  samples.mean(axis=0),
                              'Sample std':   samples.std(axis=0),
                              'P(g <= 5)':    (samples <= 5.0).mean(axis=0),
                           })
summary_5mm

,Time (years),Sample mean,Sample std,P(g <= 5)
0,0.000000,25.078865,3.559639,0.00000
1,5.263158,21.854388,3.947230,0.00000
2,10.526316,18.404606,4.383092,0.00010
3,15.789474,14.514257,4.704241,0.01666
4,21.052632,10.504953,4.983329,0.13634
5,26.315789,7.371608,5.312456,0.33720
6,31.578947,5.488843,5.722930,0.47282
7,36.842105,3.916538,6.066858,0.57630
8,42.105263,2.604511,6.279020,0.65294
9,47.368421,1.394345,6.555844,0.71094
